# `batch_monitor` — status of every native Batch job

Run **in a VWB JupyterLab app**. This notebook is **read-only**: it never
submits, cancels, or writes the ledger.

It lists Google Batch jobs in this project (smoke `eh-mini-*` / `eh-gt-*` as
well as ledger `eh-mc-*` / `eh-gt-*` shards), then optionally the GCS run
ledgers under `gs://…/batchRuns/{pipeline}/runs/`. Re-run the list cell for
a refresh. PET cannot read Cloud Logging; failed-task detail is Batch state
plus GCS worker logs / expected objects.

`JOB_NAME_PREFIX = "eh-"` hides hello-world leftovers. Clear it to see every
job. `E2_CPUS` in us-central1 is 2400 (~600 concurrent 4-vCPU minicrams).


In [ ]:
from __future__ import annotations

import os
import sys
from collections import Counter
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "monitor.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "monitor.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, gcs, monitor, poll, registry, submit  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", os.environ.get("BATCH_REGION", "us-central1"))
OUT_PREFIX = os.environ.get(
    "EH_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
)

# Job list. Prefix matches smoke + production EH ids. Empty = every Batch job.
JOB_NAME_PREFIX = os.environ.get("EH_MONITOR_PREFIX", "eh-")
PIPELINE_LABEL = os.environ.get("EH_MONITOR_PIPELINE", "")  # labels.pipeline, e.g. expansion-hunter
RUN_ID = os.environ.get("EH_RUN_ID", "")  # labels.run-id, or substring of the job id
ACTIVE_ONLY = False
ENRICH_ACTIVE = True  # describe QUEUED/SCHEDULED/RUNNING so task counts + vCPU are filled in

# Ledger scan (GCS). Independent of the Batch job list.
LEDGER_PIPELINES = ["expansion_hunter", "locityper"]
CHECK_OBJECTS = False  # ls_recursive sample outputs for RUN_ID; slow on a large cohort

# One-job drill-down (lists every task). Leave empty to skip.
DETAIL_JOB_ID = os.environ.get("EH_MONITOR_JOB", "")
LIST_TASKS = False

# us-central1 E2_CPUS was 2400 when we last checked; planning ceiling, not an API call.
E2_CPUS_QUOTA = int(os.environ.get("EH_E2_CPUS_QUOTA", "2400"))

print("REPO_ROOT", REPO_ROOT)
print("REGION", REGION)
print("JOB_NAME_PREFIX", JOB_NAME_PREFIX or "(all)")
print("RUN_ID", RUN_ID or "(all matching prefix)")
print("OUT_PREFIX", OUT_PREFIX)


## PET + project

Fails here if this is not a VWB cloud app (`GOOGLE_CLOUD_PROJECT`, `PET_SA_EMAIL`).


In [ ]:
project, sa = alloc.require_vwb_env()
print("project", project)
print("PET_SA", sa)
print("region", REGION or alloc.region())


## Batch jobs

`gcloud batch jobs list` for this project/region. Active jobs are described so
the table has task counts and an in-flight vCPU estimate (`parallelism × vCPU`
of remaining tasks). Terminal jobs stay on the cheaper list payload.


In [ ]:
region = REGION or alloc.region()
rows = monitor.list_summaries(
    project=project,
    region=region,
    name_prefix=JOB_NAME_PREFIX,
    run_id=RUN_ID,
    pipeline=PIPELINE_LABEL,
    active_only=ACTIVE_ONLY,
    enrich_active=ENRICH_ACTIVE,
)
by_state = monitor.count_by_state(rows)
vcpu = monitor.total_inflight_vcpu(rows)
print("jobs", len(rows), "by state", dict(by_state))
print(f"in-flight vCPU ~{vcpu} / {E2_CPUS_QUOTA} E2_CPUS")
shown = False
try:
    import pandas as pd
    from IPython.display import display

    if rows:
        display(pd.DataFrame(rows)[list(monitor.DISPLAY_COLUMNS)])
        shown = True
except Exception:
    pass
if not shown:
    print()
    print(monitor.format_table(rows))


## GCS run ledgers

Each `_03` / `_04` run writes `run.json` + `ledger.jsonl` under
`batchRuns/{pipeline}/runs/{RUN_ID}/`. Smoke jobs from `_02` have no ledger.


In [ ]:
bucket = OUTPUT_BUCKET_GS.rstrip("/")
for pipeline in LEDGER_PIPELINES:
    run_ids = registry.list_runs(output_bucket=bucket, pipeline=pipeline)
    print(f"{pipeline}: {len(run_ids)} run(s)", run_ids or "(none)")
    for rid in run_ids:
        paths = registry.run_paths(output_bucket=bucket, pipeline=pipeline, run_id=rid)
        meta = registry.load_run_json(paths)
        events = registry.load_events(paths)
        status = registry.load_status(paths)
        by_ev = Counter(str(e.get("event") or "") for e in events)
        by_st = Counter(f"{r.get('stage')}:{r.get('state')}" for r in status)
        print(f"  {rid}")
        print(f"    prefix {paths.prefix}")
        if meta:
            keys = ("pilot_n", "catalog", "compute", "budget_usd", "updated_at")
            shown = {k: meta[k] for k in keys if k in meta}
            print("    run.json", shown or list(meta)[:8])
        print("    events", len(events), dict(by_ev) or "(empty)")
        if status:
            print("    status.tsv", len(status), dict(by_st))
        inflight = {stage: sorted(poll.inflight_sample_ids(events, stage)) for stage in ("minicram", "genotype")}
        for stage, sids in inflight.items():
            if sids:
                print(f"    inflight {stage}: {len(sids)}", sids[:12], "..." if len(sids) > 12 else "")
        if events:
            last = events[-1]
            print("    last", last.get("ts"), last.get("event"), last.get("job_id") or "", last.get("state") or "")


## Optional: GCS objects for one `RUN_ID`

Set `CHECK_OBJECTS = True` and `RUN_ID` above. Uses `keep.csv` if present,
else `pilot.csv` / `cohort.csv`. One recursive listing of `OUT_PREFIX`.


In [ ]:
if not CHECK_OBJECTS:
    print("CHECK_OBJECTS is False — skip")
elif not RUN_ID:
    print("set RUN_ID to check sample objects")
else:
    paths = registry.run_paths(
        output_bucket=OUTPUT_BUCKET_GS.rstrip("/"),
        pipeline="expansion_hunter",
        run_id=RUN_ID,
    )
    recs = []
    source = ""
    for name in ("keep.csv", "pilot.csv", "cohort.csv"):
        uri = f"{paths.prefix}/{name}"
        if not gcs.exists(uri):
            continue
        local = Path.cwd() / "batch_monitor_scratch"
        local.mkdir(parents=True, exist_ok=True)
        path = local / name
        gcs.download_file(uri, path)
        recs = cohort.read_csv(path)
        source = uri
        break
    if not recs:
        print("no keep/pilot/cohort CSV on", paths.prefix)
    else:
        print("samples", len(recs), "from", source)
        listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
        print("objects under OUT_PREFIX", len(listing))
        for stage in ("minicram", "genotype"):
            pending, done = poll.pending_records(
                recs, stage=stage, out_prefix=OUT_PREFIX, expected=eh.expected_objects, listing=listing
            )
            print(f"  {stage}: done {len(done)} pending {len(pending)}")
            if pending:
                print("    pending ids", [r["sample_id"] for r in pending[:20]], "..." if len(pending) > 20 else "")


## Optional: one job's tasks

Set `DETAIL_JOB_ID` (and `LIST_TASKS = True`) to list every task. Skip on a
200-task shard unless you need the failing index.


In [ ]:
if not DETAIL_JOB_ID:
    print("DETAIL_JOB_ID is empty — skip")
elif not LIST_TASKS:
    job = submit.describe_job(
        job_id=DETAIL_JOB_ID, project=project, region=region, quiet=True
    )
    print(monitor.format_table([monitor.summarize_job(job)]))
    print("set LIST_TASKS = True to list each task")
else:
    detail = monitor.task_breakdown(job_id=DETAIL_JOB_ID, project=project, region=region)
    print(monitor.format_table([detail["job"]]))
    print("tasks by state", detail["by_state"])
    failed = [t for t in detail["tasks"] if t["state"] not in {"SUCCEEDED", "RUNNING"}]
    show = failed or detail["tasks"][:25]
    print("showing", len(show), "of", len(detail["tasks"]))
    for t in show:
        sec = t["seconds"]
        extra = f" {sec:.0f}s" if isinstance(sec, (int, float)) and sec is not None else ""
        print(f"  task {t['index']:>4d}  {t['state']}{extra}")
